# Identificação do projeto

**Título do projeto:** Analise do mercado de jogos na Steam  

**Integrantes do grupo:** Gustavo Delinski Tavares, Eduardo Jabur
**Turma:** Turma 6° GCL1

**Descrição breve do problema ou contexto:** Análise de dados públicos da Steam para entender quais gêneros de jogos mantêm os jogadores por mais tempo, quais gêneros mais cresceram em lançamentos e como jogos gratuitos e pagos se comparam em popularidade.

Preencha este notebook ao longo dos dois RAs. Registre decisões e interpretações em poucas frases e acrescente as células de código necessárias em cada etapa.

---

# RA1 — Data Ingestion, Cleaning and Initial Analysis

Nesta etapa, carregue, prepare e explore os dados do projeto.

## 1. Contexto e objetivo

Apresente brevemente o problema investigado e o que o grupo pretende descobrir com os dados.

**Resposta do grupo:**

A Steam é a maior loja de jogos para PC e recebe dezenas de milhares de lançamentos por ano (quase 25 mil só em 2025, segundo a base usada). Com tanta concorrência, entender quais tipos de jogo prendem os jogadores e quais gêneros estão crescendo ajuda desenvolvedores a decidir o que produzir.

O projeto vai investigar três perguntas:

- **P1:** Quais gêneros apresentam maior tempo mediano de jogo por jogador?
- **P2:** Quais gêneros tiveram maior crescimento no número de lançamentos entre 2020 e 2025?
- **P3:** Entre os 100 jogos com mais donos estimados, como gratuitos e pagos se comparam em pico de jogadores simultâneos e proporção de avaliações positivas?

Usamos o tempo de jogo como aproximação de retenção, já que a base não tem dados de jogadores voltando ao jogo ao longo do tempo. O recorte da P2 termina em 2025 porque é o último ano completo na base.

## 2. Fonte e ingestão dos dados

- Identifique a fonte dos dados, incluindo link ou referência de acesso.
- Descreva brevemente o dataset e o que seus registros representam.
- Explique como os dados foram carregados e implemente a ingestão.

**Resposta do grupo:**

**Fonte:** Steam Games Dataset, publicado no Kaggle por FronkonGames: https://www.kaggle.com/datasets/fronkongames/steam-games-dataset. Os dados foram coletados da API da Steam e do SteamSpy, até cerca de março de 2026.

**O que cada registro representa:** um item da loja Steam (na maioria jogos, mas também softwares e vídeos). São 125.855 registros e 40 colunas, com informações de preço, data de lançamento, gêneros, avaliações, donos estimados e tempo de jogo. A base vem em CSV e JSON; usamos só o CSV.

**Ingestão:** implementamos uma abordagem mista. O código tenta primeiro baixar o dataset automaticamente usando a biblioteca `kagglehub`. Caso ocorra alguma falha no download (como instabilidade ou erro na extração do arquivo), ele utiliza um fallback e lê o arquivo inserido manualmente no ambiente (`/content/games.csv`). Caso precise usar o fallback, você pode **[baixar o dataset diretamente do Kaggle clicando aqui](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset/download)** (é necessário extrair o `.zip` e fazer o upload do `games.csv` no menu lateral do Colab).

Na primeira leitura, o pandas encontrou 39 nomes de coluna para 40 valores por linha. O cabeçalho tinha as colunas Discount e DLC count unidas em "DiscountDLC count", o que deslocava todos os valores uma coluna para a esquerda. Corrigimos lendo o arquivo com a lista de nomes ajustada, sem alterar o arquivo original.

In [1]:
import pandas as pd
import kagglehub

print("Tentando baixar o dataset via kagglehub...")
try:
    caminho = kagglehub.dataset_download("fronkongames/steam-games-dataset")
    caminho_csv = f"{caminho}/games.csv"
    print(f"Dataset baixado com sucesso! Caminho: {caminho_csv}")
except Exception as e:
    print(f"Erro ao baixar via kagglehub: {e}")
    caminho_csv = "/content/games.csv"
    print(f"Usando arquivo manual como fallback: {caminho_csv}")

print("\nCarregando o CSV para o DataFrame...")
try:
    df = pd.read_csv(caminho_csv)
    print("Sucesso! DataFrame carregado.")
    print("Formato do DataFrame:", df.shape)
except Exception as e:
    print(f"Erro ao carregar o arquivo CSV: {e}")

Tentando baixar o dataset via kagglehub...
Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Dataset baixado com sucesso! Caminho: /kaggle/input/steam-games-dataset/games.csv

Carregando o CSV para o DataFrame...
Sucesso! DataFrame carregado.
Formato do DataFrame: (125855, 39)


In [2]:
df.columns.tolist()

['AppID',
 'Name',
 'Release date',
 'Estimated owners',
 'Peak CCU',
 'Required age',
 'Price',
 'DiscountDLC count',
 'About the game',
 'Supported languages',
 'Full audio languages',
 'Reviews',
 'Header image',
 'Website',
 'Support url',
 'Support email',
 'Windows',
 'Mac',
 'Linux',
 'Metacritic score',
 'Metacritic url',
 'User score',
 'Positive',
 'Negative',
 'Score rank',
 'Achievements',
 'Recommendations',
 'Notes',
 'Average playtime forever',
 'Average playtime two weeks',
 'Median playtime forever',
 'Median playtime two weeks',
 'Developers',
 'Publishers',
 'Categories',
 'Genres',
 'Tags',
 'Screenshots',
 'Movies']

In [3]:
colunas = pd.read_csv(caminho_csv, nrows=0).columns.tolist()

i = colunas.index("DiscountDLC count")
colunas[i:i+1] = ["Discount", "DLC count"]

df = pd.read_csv(caminho_csv, header=None, skiprows=1, names=colunas)
print(df.shape)

(125855, 40)


## 3. Exploração inicial dos dados

Investigue a quantidade de registros e colunas, os tipos de dados, os valores ausentes e outras características relevantes da base.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125855 non-null  int64  
 1   Name                        125854 non-null  object 
 2   Release date                125855 non-null  object 
 3   Estimated owners            125855 non-null  object 
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  int64  
 6   Price                       125855 non-null  float64
 7   Discount                    125855 non-null  int64  
 8   DLC count                   125855 non-null  int64  
 9   About the game              117393 non-null  object 
 10  Supported languages         125855 non-null  object 
 11  Full audio languages        125855 non-null  object 
 12  Reviews                     12181 non-null   object 
 13  Header image  

In [5]:
df["Estimated owners"].value_counts()

,count
Estimated owners,
0 - 20000,75705
0 - 0,24579
20000 - 50000,11397
50000 - 100000,5355
100000 - 200000,3454
200000 - 500000,2856
500000 - 1000000,1155
1000000 - 2000000,729
2000000 - 5000000,405


In [6]:
df.isna().sum().sort_values(ascending=False)

,0
Movies,125855
Score rank,125815
Metacritic url,121597
Reviews,113674
Notes,102545
Website,75335
Support url,70560
Tags,42502
Support email,22630
Categories,8966


In [7]:
print((df["Median playtime forever"] == 0).mean())
print((df["Estimated owners"] == "0 - 0").mean())
print((df["Price"] == 0).mean())

0.7920384569544316
0.1952961741686862
0.21183902109570538


In [8]:
df["Release date"].sample(15, random_state=1)

,Release date
55959,"Apr 11, 2024"
42009,"Dec 22, 2023"
51173,"Nov 8, 2024"
26675,"Dec 20, 2016"
51005,"Jul 24, 2017"
112101,"Oct 23, 2024"
34898,"Mar 27, 2019"
90039,"Jul 12, 2025"
22813,"Oct 1, 2025"
7139,"Oct 23, 2025"


In [9]:
datas = pd.to_datetime(df["Release date"], format="%b %d, %Y", errors="coerce")
print(datas.isna().sum())
print(datas.min())
print(datas.max())

0
1997-06-30 00:00:00
2026-12-01 00:00:00


In [10]:
datas.dt.year.value_counts().sort_index().tail(10)

,count
Release date,
2017,5922
2018,7461
2019,7243
2020,8804
2021,11067
2022,12288
2023,14598
2024,20033
2025,24973


In [11]:
datas[datas.dt.year >= 2025].dt.to_period("M").value_counts().sort_index()

,count
Release date,
2025-01,1680
2025-02,1629
2025-03,2087
2025-04,2092
2025-05,2040
2025-06,1999
2025-07,2264
2025-08,2188
2025-09,2111


In [12]:
df["Genres"].sample(10, random_state=1)

,Genres
55959,"Casual,Indie"
42009,NaN
51173,"Casual,Strategy,Free To Play,Early Access"
26675,Simulation
51005,"Action,Indie"
112101,"Casual,Indie,Strategy,Early Access"
34898,"Casual,Indie"
90039,"Action,Adventure,Casual,Simulation"
22813,"Action,Adventure,RPG,Free To Play"
7139,"Casual,Indie"


In [13]:
generos = df["Genres"].str.split(",").explode()
generos.value_counts()

,count
Genres,
Indie,82884
Casual,51677
Action,47344
Adventure,46293
Simulation,24912
Strategy,23058
RPG,21577
Free To Play,12605
Early Access,11572


In [14]:
generos_software = ["Utilities", "Design & Illustration", "Animation & Modeling",
                    "Video Production", "Game Development", "Audio Production",
                    "Software Training", "Web Publishing", "Photo Editing",
                    "Education", "Accounting", "Movie", "Documentary",
                    "Episodic", "Short", "Tutorial", "360 Video"]

rotulos_nao_genero = ["Violent", "Gore", "Nudity", "Sexual Content",
                      "Early Access", "Free To Play"]

lista = df["Genres"].dropna().str.split(",")

tem_software = lista.apply(lambda g: any(x in generos_software for x in g))
generos_de_jogo = lista.apply(lambda g: [x for x in g if x not in generos_software and x not in rotulos_nao_genero])
tem_jogo = generos_de_jogo.apply(len) > 0

print("Só software/vídeo:", (tem_software & ~tem_jogo).sum())
print("Misturam jogo e software:", (tem_software & tem_jogo).sum())
print("Sem nenhum gênero de jogo (só rótulos):", (~tem_software & ~tem_jogo).sum())

Só software/vídeo: 1491
Misturam jogo e software: 410
Sem nenhum gênero de jogo (só rótulos): 220


In [15]:
df.nlargest(5, "Median playtime forever")[["Name", "Median playtime forever"]]

,Name,Median playtime forever
27968,Avem33,3429544
59930,iTownGamePlay UNIVERSE,1714283
67121,Letters From a Rainy Day -Oceans and Lace-,359674
63635,爱人 Lover,322983
21667,Sisyphus Reborn,222407


In [16]:
print("AppID duplicado:", df["AppID"].duplicated().sum())
print("Nome duplicado:", df["Name"].duplicated().sum())
print("Nome + Dev + Data duplicado:", df.duplicated(subset=["Name", "Developers", "Release date"]).sum())

AppID duplicado: 0
Nome duplicado: 1210
Nome + Dev + Data duplicado: 111


**Resposta do grupo:**

O dataset possui 125.855 registros, cada linha é um item da loja Steam, e 40 colunas após a correção do cabeçalho. Os tipos incluem números inteiros (AppID E Peak CCU), decimais (Price), booleanos (Windows, Linux) e texto (Name, Genres, Release date, Estimated owners).

Principais problemas e características observadas:

1. Cabeçalho: as colunas Discount e DLC count vinham unidas em DiscountDLC count, deslocando todos os valores uma coluna para a esquerda. Corrigido na ingestão.

2. Valores ausentes: entre as colunas usadas no projeto, Genres tem 8.423 vazios e Name tem 1. Outras colunas com muitos vazios (Website, Metacritic url, Support email etc.) não são usadas nas perguntas.

3. Zeros que indicam falta de dado: 79% dos registros têm Median playtime forever = 0 e 19,5% têm Estimated owners = "0 - 0". Interpretamos como ausência de dado (o SteamSpy não estima jogos com poucos jogadores), não como valor real.

4. Zeros no preço: 21% dos registros têm Price = 0, mas notamos que nem todos são gratuitos (jogos legados e deslistados também recebem valor 0 na base de dados).

5. Estimated owners: está como texto em faixas (ex: "0 - 20000"), não como número.

6. Release date: está como texto no formato "Mon DD, YYYY" e converte para data sem falhas. A coleta vai até cerca de março de 2026, então 2026 está incompleto. Existem datas futuras (jogos anunciados, ainda não lançados).

7. Genres: vários gêneros por jogo em um único texto separado por vírgula. Há itens que não são jogos: 1.491 possuem apenas gêneros de software/vídeo, 410 misturam gêneros de jogo e de software, e 220 possuem apenas rótulos que não são gênero (ex: "Free To Play", "Early Access", "Violent").

8. Outliers: alguns jogos têm tempo
mediano de jogo implausível (ex: Avem33, cerca de 6,5 anos ininterruptos), provavelmente jogos mantidos abertos em segundo plano.

9. Duplicados: não há AppID repetido.
 Nomes repetidos são, em geral, jogos diferentes com o mesmo nome. Porém, há registros com mesmo nome, desenvolvedor e data de lançamento, que representam o mesmo jogo cadastrado mais de uma vez.

## 4. Limpeza e preparação dos dados

Realize a limpeza necessária ao projeto. Verifique valores ausentes, duplicados e inconsistências e trate os problemas encontrados, quando necessário.

In [17]:
df_limpo = df.copy()

In [18]:
print("Tipo antes:", df_limpo["Estimated owners"].dtype)
print("Registros '0 - 0':", (df_limpo["Estimated owners"] == "0 - 0").sum())

faixas = df_limpo["Estimated owners"].str.split(" - ", expand=True).astype(int)
df_limpo["owners_min"] = faixas[0]
df_limpo["owners_max"] = faixas[1]
df_limpo["owners_medio"] = (faixas[0] + faixas[1]) / 2

sem_dado = df_limpo["Estimated owners"] == "0 - 0"
df_limpo.loc[sem_dado, ["owners_min", "owners_max", "owners_medio"]] = None

print("Tipo depois:", df_limpo["owners_medio"].dtype)
print("Vazios em owners_medio:", df_limpo["owners_medio"].isna().sum())
df_limpo[["Estimated owners", "owners_min", "owners_max", "owners_medio"]].sample(5, random_state=1)

Tipo antes: object
Registros '0 - 0': 24579
Tipo depois: float64
Vazios em owners_medio: 24579


,Estimated owners,owners_min,owners_max,owners_medio
55959,20000 - 50000,20000.0,50000.0,35000.0
42009,0 - 20000,0.0,20000.0,10000.0
51173,0 - 20000,0.0,20000.0,10000.0
26675,0 - 20000,0.0,20000.0,10000.0
51005,20000 - 50000,20000.0,50000.0,35000.0


In [19]:
col = "Median playtime forever"
print("Zeros antes:", (df_limpo[col] == 0).sum())

df_limpo[col] = df_limpo[col].mask(df_limpo[col] == 0)

print("Zeros depois:", (df_limpo[col] == 0).sum())
print("Vazios depois:", df_limpo[col].isna().sum())

Zeros antes: 99682
Zeros depois: 0
Vazios depois: 99682


In [20]:
print("Tipo antes:", df_limpo["Release date"].dtype)

df_limpo["Release date"] = pd.to_datetime(df_limpo["Release date"], format="%b %d, %Y")

print("Tipo depois:", df_limpo["Release date"].dtype)

Tipo antes: object
Tipo depois: datetime64[ns]


In [21]:
so_software = tem_software & ~tem_jogo
print("Itens só software antes:", so_software.sum())

df_limpo = df_limpo.drop(index=so_software[so_software].index)

print("Registros depois:", len(df_limpo))

Itens só software antes: 1491
Registros depois: 124364


In [22]:
print("Itens sem nome antes:", df_limpo["Name"].isna().sum())
df_limpo = df_limpo.dropna(subset=["Name"])
print("Itens sem nome depois:", df_limpo["Name"].isna().sum())

Itens sem nome antes: 1
Itens sem nome depois: 0


In [23]:
chave = ["Name", "Developers", "Release date"]
print("Duplicados antes:", df_limpo.duplicated(subset=chave).sum())

df_limpo = df_limpo.sort_values(["owners_medio", "Positive"], ascending=False)
df_limpo = df_limpo.drop_duplicates(subset=chave, keep="first")

print("Duplicados depois:", df_limpo.duplicated(subset=chave).sum())
print("Registros depois:", len(df_limpo))
df_limpo[df_limpo["Name"] == "Assassin's Creed 2"][["AppID", "Estimated owners"]]

Duplicados antes: 111
Duplicados depois: 0
Registros depois: 124252


,AppID,Estimated owners
63287,33230,2000000 - 5000000


**Resposta do grupo:**

Antes de qualquer correção, criamos uma cópia (df_limpo) para manter os dados originais intactos e permitir comparar antes e depois.

- **D1. Estimated owners:** convertido de faixa de texto para número. Usamos o ponto médio da faixa (owners_medio) e mantivemos também o mínimo, o máximo e a coluna original. O ponto médio é uma aproximação, mas permite ordenar e comparar jogos. Os 24.579 registros com "0 - 0" viraram vazio, por serem falta de dado.
- **D2. Median playtime forever:** os 99.682 valores iguais a 0 viraram vazio. Removê-los apagaria 79% da base e prejudicaria a P2, que não depende de tempo de jogo. Como vazio, eles ficam fora do cálculo da mediana na P1.
- **D3. Release date:** convertida de texto para data, permitindo extrair o ano e filtrar períodos.
- **D4. Itens que não são jogos:** removidos os 1.491 itens com apenas gêneros de software ou vídeo. Os 410 que misturam gêneros de jogo e de software foram mantidos, pois são cerca de 0,3% da base e uma revisão manual não seria viável. Limitação: alguns softwares marcados com gênero de jogo continuam na base.
- **D5. Registro sem nome:** removido (1 registro), pois não é possível identificar o jogo.
- **D6. Duplicados:** removidos 111 registros repetidos por nome, desenvolvedor e data. Antes de remover, ordenamos pelo número de donos e de avaliações positivas, para manter a cópia com dados reais (ex: Assassin's Creed 2 manteve a versão com 2 a 5 milhões de donos).
- **Mantidos sem alteração:** preços iguais a 0 (são jogos gratuitos), datas futuras (não entram no recorte 2020 a 2025 da P2) e tempos de jogo extremos (poucos casos, e a P1 usa mediana, que é pouco afetada por valores extremos).

## 5. Transformação dos dados

Realize as transformações necessárias à análise, como criação de colunas, filtros, agrupamentos, conversões ou junções, quando aplicáveis.

In [24]:
df_limpo["generos_jogo"] = generos_de_jogo.str.join(",")
df_limpo["generos_jogo"] = df_limpo["generos_jogo"].replace("", None)


df_limpo["gratuito"] = (df_limpo["Price"] == 0) & (df_limpo["Genres"].str.contains("Free To Play", na=False))
df_limpo["ano_lancamento"] = df_limpo["Release date"].dt.year

print("Jogos sem gênero de jogo:", df_limpo["generos_jogo"].isna().sum())
df_limpo[["Genres", "generos_jogo", "gratuito", "ano_lancamento"]].sample(5, random_state=1)

Jogos sem gênero de jogo: 8642


,Genres,generos_jogo,gratuito,ano_lancamento
111934,Action,Action,False,2025
121925,"Casual,Indie,Simulation","Casual,Indie,Simulation",False,2025
46561,"Adventure,Indie","Adventure,Indie",False,2025
112489,Simulation,Simulation,False,2023
1854,"Casual,Indie,RPG,Strategy,Early Access","Casual,Indie,RPG,Strategy",False,2022


In [25]:
colunas_finais = ["AppID", "Name", "Release date", "ano_lancamento",
                  "Price", "gratuito",
                  "Estimated owners", "owners_min", "owners_max", "owners_medio",
                  "Peak CCU", "Positive", "Negative", "Median playtime forever",
                  "Genres", "generos_jogo", "Developers", "Publishers"]

df_final = df_limpo[colunas_finais].sort_values("AppID").reset_index(drop=True)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124252 entries, 0 to 124251
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   AppID                    124252 non-null  int64         
 1   Name                     124252 non-null  object        
 2   Release date             124252 non-null  datetime64[ns]
 3   ano_lancamento           124252 non-null  int32         
 4   Price                    124252 non-null  float64       
 5   gratuito                 124252 non-null  bool          
 6   Estimated owners         124252 non-null  object        
 7   owners_min               99883 non-null   float64       
 8   owners_max               99883 non-null   float64       
 9   owners_medio             99883 non-null   float64       
 10  Peak CCU                 124252 non-null  int64         
 11  Positive                 124252 non-null  int64         
 12  Negative        

In [26]:
print("Registros:", len(df), "->", len(df_final))
print("Colunas:", df.shape[1], "->", df_final.shape[1])
print("AppID duplicado no final:", df_final["AppID"].duplicated().sum())
print("Duplicados (nome + dev + data) no final:", df_final.duplicated(subset=["Name", "Developers", "Release date"]).sum())

Registros: 125855 -> 124252
Colunas: 40 -> 18
AppID duplicado no final: 0
Duplicados (nome + dev + data) no final: 0


In [27]:
df_final.to_csv("steam_games_limpo.csv", index=False)

from google.colab import files
files.download("steam_games_limpo.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Resposta do grupo:**

- **generos_jogo:** lista de gêneros sem os gêneros de software e sem rótulos que não são gênero (Violent, Gore, Nudity, Sexual Content, Early Access, Free To Play). É a coluna usada na P1 e na P2. A coluna Genres original foi mantida.
- **gratuito:** indica se o jogo é realmente gratuito (preço igual a 0 E possui a tag "Free To Play" nos gêneros). Isso evita classificar jogos legados (como GTA V Legacy) como gratuitos. Usada na P3.
- **ano_lancamento:** ano extraído da data de lançamento. Usada na P2.
- **Seleção de colunas:** mantivemos apenas as 18 colunas usadas nas perguntas e no RA2. Descrições, imagens, links e e-mails foram descartados por não serem usados.
- **Arquivo final:** steam_games_limpo.csv, com 124.252 registros e 18 colunas.

## 7. Considerações do RA1

Registre brevemente:

- O que o grupo descobriu sobre os dados.
- Quais dificuldades ou limitações encontrou.
- Como os dados preparados poderão ser utilizados nas próximas etapas.

**Resposta do grupo:**

**O que descobrimos:** a base parecia completa olhando só os valores vazios, mas muitos problemas estavam escondidos. O cabeçalho deslocava todas as colunas, 79% dos tempos de jogo eram zero por falta de dado, e havia softwares e registros duplicados misturados aos jogos. Também descobrimos que um preço igual a 0 não significa obrigatoriamente que o jogo é gratuito (jogos removidos ou legados ficam com preço zero), sendo necessário exigir a tag "Free To Play" para confirmá-los. Por fim, percebemos que 2026 está incompleto, o que distorceria qualquer comparação por ano.

**Dificuldades e limitações:**
- O número de donos é apenas uma faixa estimada pelo SteamSpy, e o ponto médio é uma aproximação.
- Só cerca de 26 mil jogos têm tempo de jogo informado, então a P1 representa principalmente jogos com mais jogadores.
- A separação entre jogo e software depende dos gêneros marcados pelo próprio desenvolvedor, que nem sempre são confiáveis.
- O tempo de jogo é uma aproximação de retenção, não uma medida direta.

**Próximas etapas:** o arquivo steam_games_limpo.csv será a base das análises. No RA2, ele será cruzado pelo AppID com dados reais de jogadores online obtidos pela API da Steam, para acompanhar o número de jogadores por gênero e por tipo (gratuito ou pago) em tempo real.

---

# RA2 — Streaming Data Processing and Time Windows

Utilize os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A — Dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.
- **Cenário B — Dados em streaming:** utilize sua própria fonte de dados contínuos, sem necessidade de simulação.

## 8. Objetivo da análise em streaming

Responda brevemente:

- O que o grupo deseja monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

**Resposta do grupo:**

- **O que desejamos monitorar:** O número de jogadores simultâneos online em tempo real na Steam para uma amostra dos jogos mais populares da base limpa (`df_final`), cruzando as leituras ao vivo da API com o modelo de negócio (gratuito vs. pago).
- **Informação que esperamos obter:** Comparar em tempo real o volume total e a média de jogadores ativos entre jogos gratuitos e pagos dentro de janelas de tempo contínuas, observando como os indicadores são atualizados a cada nova leitura recebida.

## 9. Fonte de dados e processamento

- Informe o cenário escolhido: dados reais em streaming ou simulação.
- Explique brevemente como os novos registros chegam.
- Implemente o processamento em tempo real.
- Demonstre o recebimento de novos registros.

**Resposta do grupo:**

- **Cenário escolhido:** Cenário B — Dados reais em streaming via API pública da Steam (`ISteamUserStats/GetNumberOfCurrentPlayers/v1/`).
- **Como os novos registros chegam:** Selecionamos 10 grandes jogos da base `df_final` (5 gratuitos e 5 pagos). A função `coletar_lote_steam()` consulta a API em tempo real, registra o `player_count` e o `timestamp` do momento da coleta e salva um arquivo JSON na pasta `stream_input/`.
- **Processamento em tempo real:** O Spark Structured Streaming (`spark.readStream`) monitora a pasta `stream_input/`, detecta cada novo arquivo JSON instantaneamente e faz o *join* com os metadados do RA1 (`AppID`, `Name`, `gratuito` e `generos_jogo`).

In [28]:
!pip install -q pyspark

import os, time, json, shutil, requests
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType
from pyspark.sql.functions import col, window, sum as _sum, avg, count, round as _round

spark = SparkSession.builder \
    .appName("Steam_Streaming_RA2") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

if 'df_final' not in locals():
    df_final = pd.read_csv("steam_games_limpo.csv")

top_gratuitos = df_final[df_final["gratuito"] == True].sort_values(["owners_medio", "Peak CCU"], ascending=False).head(5)
top_pagos = df_final[df_final["gratuito"] == False].sort_values(["owners_medio", "Peak CCU"], ascending=False).head(5)
amostra_pd = pd.concat([top_gratuitos, top_pagos])

lista_appids = amostra_pd["AppID"].tolist()
df_meta_spark = spark.createDataFrame(amostra_pd[["AppID", "Name", "gratuito", "generos_jogo"]])

print("10 jogos selecionados para monitoramento em tempo real (5 Gratuitos e 5 Pagos):")
df_meta_spark.show(10, truncate=False)

10 jogos selecionados para monitoramento em tempo real (5 Gratuitos e 5 Pagos):
+-------+-------------------------+--------+--------------------------------------+
|AppID  |Name                     |gratuito|generos_jogo                          |
+-------+-------------------------+--------+--------------------------------------+
|730    |Counter-Strike 2         |true    |Action                                |
|570    |Dota 2                   |true    |Action,Strategy                       |
|578080 |PUBG: BATTLEGROUNDS      |true    |Action,Adventure,Massively Multiplayer|
|1172470|Apex Legends™            |true    |Action,Adventure                      |
|440    |Team Fortress 2          |true    |Action                                |
|271590 |Grand Theft Auto V Legacy|false   |Action,Adventure                      |
|1938090|Call of Duty®            |false   |Action                                |
|550    |Left 4 Dead 2            |false   |Action                              

In [29]:
for q in spark.streams.active:
    q.stop()

pasta_stream = "stream_input"
shutil.rmtree(pasta_stream, ignore_errors=True)
os.makedirs(pasta_stream, exist_ok=True)


def consultar_api(appid):
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={appid}"
    try:
        r = requests.get(url, timeout=3).json().get("response", {})
        if r.get("result") == 1:
            return {"AppID": int(appid), "player_count": int(r.get("player_count", 0))}
    except Exception:
        pass
    return None


def coletar_lote_steam(nome_lote, ts_custom=None):
    agora = ts_custom if ts_custom else datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with ThreadPoolExecutor(max_workers=10) as executor:
        resultados = list(executor.map(consultar_api, lista_appids))

    registros = [{**r, "timestamp": agora} for r in resultados if r is not None]
    caminho = os.path.join(pasta_stream, f"lote_{nome_lote}_{int(time.time()*1000)}.json")
    with open(caminho, "w") as f:
        for reg in registros:
            f.write(json.dumps(reg) + "\n")
    print(f"[Lote '{nome_lote}'] {len(registros)} jogos coletados da API da Steam (timestamp: {agora})")


schema_steam = StructType([
    StructField("AppID", IntegerType(), True),
    StructField("player_count", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])

stream_enriquecido = spark.readStream \
    .schema(schema_steam) \
    .json(pasta_stream) \
    .join(df_meta_spark, on="AppID", how="inner")

query_brutos = stream_enriquecido.writeStream \
    .format("memory") \
    .queryName("registros_recebidos") \
    .outputMode("append") \
    .start()

coletar_lote_steam("inicial")
query_brutos.processAllAvailable()

spark.sql("SELECT timestamp, AppID, Name, gratuito, player_count FROM registros_recebidos ORDER BY player_count DESC").show(10, truncate=False)

[Lote 'inicial'] 10 jogos coletados da API da Steam (timestamp: 2026-10-21 15:15:10)
+-------------------+-------+-------------------------+--------+------------+
|timestamp          |AppID  |Name                     |gratuito|player_count|
+-------------------+-------+-------------------------+--------+------------+
|2026-10-21 15:15:10|730    |Counter-Strike 2         |true    |1177224     |
|2026-10-21 15:15:10|570    |Dota 2                   |true    |837646      |
|2026-10-21 15:15:10|578080 |PUBG: BATTLEGROUNDS      |true    |648826      |
|2026-10-21 15:15:10|1172470|Apex Legends™            |true    |165054      |
|2026-10-21 15:15:10|1938090|Call of Duty®            |false   |120610      |
|2026-10-21 15:15:10|440    |Team Fortress 2          |true    |73546       |
|2026-10-21 15:15:10|271590 |Grand Theft Auto V Legacy|false   |63944       |
|2026-10-21 15:15:10|1623730|Palworld                 |false   |49668       |
|2026-10-21 15:15:10|550    |Left 4 Dead 2            |fa

## 10. Aplicação de janelas temporais

- Utilize uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informe o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implemente uma agregação coerente com a análise, como contagem, soma ou média.
- Explique brevemente o que será calculado em cada intervalo.

O foco é demonstrar a compreensão do conceito; não é necessário justificar tecnicamente a duração escolhida.

**Resposta do grupo:**

- **Tipo de janela:** *Tumbling window* (janelas fixas, sem sobreposição).
- **Duração da janela:** 10 segundos (sem deslocamento, pois as janelas não se sobrepõem).
- **Agregação implementada:** Agrupamento pela janela de 10 segundos (`window(col("timestamp"), "10 seconds")`) e pela coluna `gratuito`.
- **O que é calculado em cada intervalo:** A quantidade de registros recebidos na janela (`leituras`) e a média de jogadores por categoria (`media_jogadores`). O uso de contagem e média (em vez de soma) garante que não contaremos o mesmo jogador duas vezes ao processar várias atualizações do mesmo jogo na mesma janela.

In [30]:
for q in spark.streams.active:
    q.stop()

shutil.rmtree(pasta_stream, ignore_errors=True)
os.makedirs(pasta_stream, exist_ok=True)

stream_enriquecido = spark.readStream \
    .schema(schema_steam) \
    .json(pasta_stream) \
    .join(df_meta_spark, on="AppID", how="inner")

janela_tumbling = stream_enriquecido \
    .groupBy(
        window(col("timestamp"), "10 seconds"),
        col("gratuito")
    ) \
    .agg(
        count("AppID").alias("leituras"),
        _round(avg("player_count"), 0).alias("media_jogadores")
    )

query_janela = janela_tumbling.writeStream \
    .format("memory") \
    .queryName("tabela_janelas") \
    .outputMode("complete") \
    .start()

def mostrar_tabela_janelas():
    spark.sql("""
        SELECT
            date_format(window.start, 'HH:mm:ss') AS inicio_janela,
            date_format(window.end, 'HH:mm:ss') AS fim_janela,
            CASE WHEN gratuito THEN 'Gratuito' ELSE 'Pago' END AS tipo_jogo,
            leituras,
            media_jogadores
        FROM tabela_janelas
        ORDER BY inicio_janela, tipo_jogo
    """).show(truncate=False)

print("Streaming com janela Tumbling de 10s ativo! Pronto para a demonstração na Seção 11.")

Streaming com janela Tumbling de 10s ativo! Pronto para a demonstração na Seção 11.


## 11. Resultados e interpretação

Apresente os resultados do processamento e demonstre como a análise se comporta com a chegada de novos registros.

In [31]:
base_minuto = datetime.now().strftime("%Y-%m-%d %H:%M")

coletar_lote_steam("Lote_1", ts_custom=f"{base_minuto}:02")
query_janela.processAllAvailable()

print("=== TABELA ANTES (Apenas Lote 1 recebido na Janela 1) ===")
mostrar_tabela_janelas()

[Lote 'Lote_1'] 10 jogos coletados da API da Steam (timestamp: 2026-10-21 15:15:02)
=== TABELA ANTES (Apenas Lote 1 recebido na Janela 1) ===
+-------------+----------+---------+--------+---------------+
|inicio_janela|fim_janela|tipo_jogo|leituras|media_jogadores|
+-------------+----------+---------+--------+---------------+
|15:15:00     |15:15:10  |Gratuito |5       |580459.0       |
|15:15:00     |15:15:10  |Pago     |5       |55274.0        |
+-------------+----------+---------+--------+---------------+



In [32]:
coletar_lote_steam("Lote_2", ts_custom=f"{base_minuto}:05")
query_janela.processAllAvailable()

print("=== TABELA DEPOIS DO LOTE 2 (Mesma janela atualizada: leituras sobem de 5 para 10) ===")
mostrar_tabela_janelas()

[Lote 'Lote_2'] 10 jogos coletados da API da Steam (timestamp: 2026-10-21 15:15:05)
=== TABELA DEPOIS DO LOTE 2 (Mesma janela atualizada: leituras sobem de 5 para 10) ===
+-------------+----------+---------+--------+---------------+
|inicio_janela|fim_janela|tipo_jogo|leituras|media_jogadores|
+-------------+----------+---------+--------+---------------+
|15:15:00     |15:15:10  |Gratuito |10      |580459.0       |
|15:15:00     |15:15:10  |Pago     |10      |55274.0        |
+-------------+----------+---------+--------+---------------+



In [33]:
coletar_lote_steam("Lote_3", ts_custom=f"{base_minuto}:12")
query_janela.processAllAvailable()

print("=== TABELA DEPOIS DO LOTE 3 (Nova janela temporal criada automaticamente pelo Spark) ===")
mostrar_tabela_janelas()

query_janela.stop()

[Lote 'Lote_3'] 10 jogos coletados da API da Steam (timestamp: 2026-10-21 15:15:12)
=== TABELA DEPOIS DO LOTE 3 (Nova janela temporal criada automaticamente pelo Spark) ===
+-------------+----------+---------+--------+---------------+
|inicio_janela|fim_janela|tipo_jogo|leituras|media_jogadores|
+-------------+----------+---------+--------+---------------+
|15:15:00     |15:15:10  |Gratuito |10      |580459.0       |
|15:15:00     |15:15:10  |Pago     |10      |55274.0        |
|15:15:10     |15:15:20  |Gratuito |5       |580459.0       |
|15:15:10     |15:15:20  |Pago     |5       |54846.0        |
+-------------+----------+---------+--------+---------------+



**Resposta do grupo:**

- **Significado dos resultados:** Mesmo entre os 10 jogos de maior alcance na Steam, os jogos gratuitos apresentam uma média de jogadores simultâneos muito superior aos jogos pagos em tempo real.
- **Mudanças observadas entre o "Antes" e o "Depois":**
  1. **Antes (Lote 1):** A tabela exibe a primeira janela (`:00` a `:10`) com 5 leituras para jogos gratuitos e 5 para pagos.
  2. **Depois (Lote 2 na mesma janela):** Com a chegada de novos dados dentro do mesmo intervalo de 10 segundos, a janela existente foi atualizada em tempo real — a coluna `leituras` dobrou de 5 para 10 em cada grupo, e a média foi recalculada.
  3. **Depois (Lote 3 em nova janela):** Quando os novos registros chegaram após o limite de 10 segundos (`:12`), a janela anterior permaneceu intacta e o Spark abriu uma nova linha na tabela para a janela `:10` a `:20`, isolando as novas leituras.
- **Notas técnicas:**
  - Na demonstração, fixamos o timestamp de cada lote (`:02`, `:05` e `:12`) para garantir que os lotes 1 e 2 caíssem na mesma janela e o lote 3 na seguinte, facilitando a visualização didática. Os números de jogadores continuam sendo leituras reais da API no momento da execução.
  - Em execuções muito próximas, a média de jogadores pode não mudar. Isso acontece porque a API da Steam não atualiza o número de jogadores a cada segundo; ela devolve um valor que muda apenas a cada poucos minutos.

## 12. Considerações do RA2

Responda brevemente:

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?

Na última resposta, o grupo pode reconhecer que o streaming foi utilizado apenas como exercício didático, se esse for o caso.

**Resposta do grupo:**

- **O que foi possível observar com o processamento em streaming:** Vimos na prática como o Spark Structured Streaming ingere novos arquivos JSON assim que chegam no diretório e os cruza instantaneamente com a tabela limpa do RA1 (`df_final`), atualizando a tabela de resultados em memória sem intervenção manual.
- **Qual foi a contribuição das janelas temporais para a análise:** A janela *Tumbling* permitiu separar o fluxo contínuo da API da Steam em blocos fixos de tempo, mostrando tanto a atualização de métricas dentro de uma janela ativa quanto a criação de novas janelas conforme o tempo avança.
- **Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?** Sim. Em uma aplicação real (com janelas de 5 ou 15 minutos), essa arquitetura permitiria monitorar o impacto imediato de lançamentos, atualizações (*patches*) ou promoções no número de jogadores online por gênero ou modelo de negócio.